## 데이터 구축 방안

1. 시군구 코드, 용도 코드 변환 규칙 작성
    1. 시군구(주민등록인구와 일치) 단위 집계
        1. 변환 규칙 1: 오류만 정정, 변환 안함
        2. 변환 규칙 2: 자치구 단위 집계, 자치구 단위 통계 비교 목적으로 사용
    2. 용도 통계분류(5개) 단위 집계를 위한 용도 변환 규칙 작성
2. 층별개요 데이터 정제
    1. 시군구*코드, 층*구분*코드, 주*용도\_코드, 면적(㎡) 컬럼의 값이 유효한 경우만 추출
3. 층별개요와 표제부 데이터 연계 및 정제
    1. 표제부 데이터 정제
        1. 표제부에서 주건축물만 추출(부속건축물, 미기재 제거)
    2. 표제부 PK (관리\_건축물대장\_PK) 사용하여 층별개요에 표제부 데이터 연계
        1. inner join 층별개요에 유효한 주건축물 표제부 PK가 있으면 n:1 연계
    3. 표제부 연면적이 0보다 크면서, 층 면적이 연면적보다 큰 경우 제거
4. 시군구(주민등록인구와 일치), 용도(통계분류, 5개) 단위 집계


In [40]:
import polars as pl
from pathlib import Path
from pprint import pprint

df_master = pl.scan_parquet("data/건축물대장_기본개요_2022년_12월/*")
df_dong = pl.scan_parquet("data/건축물대장_표제부_2022년_12월/*")
df_floor = pl.scan_parquet("data/건축물대장_층별개요_2022년_12월/*")

df_sgg = pl.scan_csv("data/code_sgg.csv", dtypes={"시군구코드": str})

## 시군구, 용도 변환 규칙


생산량(허가, 착공, 준공) 집계 기준

-   가설건축물, 기타 용도 제외

3. 연면적 : 층 용도별 면적의 합
4. 용도 : 층별 용도

-   층 용도별 면적의 합
-   건축인허가
    -   면적은 층별 층면적 사용
    -   용도는 층별 주용도 사용
    -   시군구는 층별 사용(어차피 기본에서 온 것)
-   주택인허가
    -   면적은 층별 층면적 사용
    -   용도는 층별 용도 사용
    -   시군구는 층별 사용(어차피 기본에서 온 것)

---

건축물 : 토지에 정착하는 공작물중 지붕과 기둥 또는 벽이 있는 것과 이에 부수되는 시설물, 지하 또는 고가의 공작물에 설치하는 사무소, 공연장, 점포, 차고, 창고 기타 대통령령이 정하는 것

건축물의 용도 : 건축물의 종류(단독주택, 공동주택 등)를 유사한 구조, 이용목적 및 형태별로 묶어 분류한 것을 말함

주거용 : 공동주택, 단독주택

상업용 : 판매및영업시설,제1종근린생활시설,제2종근린생활시설,판매시설,운수시설,업무시설,숙박시설,위락시설,위험물저장및처리시설,자동차관련시설,야영장시설

농수산용 : 동,식물관련시설

공업용 : 공장

공공용 : 공공용시설,교정및군사시설,방송통신시설,발전시설

교육및사회용 : 교육연구및복지시설,문화및집회시설,종교시설,의료시설,교육연구시설,노유자시설,수련시설,운동시설,묘지관련시설,관광휴게시설,장례시설

기타 : 기타,창고시설,분뇨.쓰레기처리시설,가설건축물시설,자원순환관련시설

https://www.index.go.kr/unity/potal/main/EachDtlPageDetail.do?idx_cd=1226

---

※ 위 자료에서 분류하는 통계용 건축물의 용도는 아래와 같습니다.

[용도별 건축물 분류(통계용)]

-   주거용 : 단독, 다가구, 아파트, 연립, 다세대, 기타(다중주택, 공관, 기숙사 등)

-   상업용 : 근린생활, 판매, 업무, 숙박, 위락, 운수, 자동차관련시설 등

-   공업용 : 공장

-   교육및사회용 : 문화집회시설(극장 등), 종교시설, 의료시설, 교육연구시설(학교 등), 노유자시설, 수련시설, 운동시설, 관광휴게시설, 묘지관련시설 및 장례시설 등

-   기타 : 농수산용(축사, 온실), 공공용(공공청사, 방송국), 창고 등

[상업용 건축물 분류(통계용)]

-   제1종근린생활시설 : 소매점, 휴게음식점, 이용원, 의원 등

-   제2종근린생활시설 : 공연장, 금융업소, 제조업소, 고시원 등

-   판매시설 : 도매시장, 소매시장, 상점 등

-   업무시설 : 공공업무시설, 일반업무시설(사무소, 오피스텔 등)

-   기타 : 위락시설, 숙박시설, 운수시설, 자동차관련시설 등

[출처] 전국 건축물 총 7,354,340동… 연면적 41억 3천만㎡|작성자 국토교통부
https://blog.naver.com/mltmkr/223032348016


In [41]:
use_agg = {
    "01000": "01",
    "01001": "01",
    "01002": "01",
    "01003": "01",
    "01004": "01",
    "02000": "02",
    "02001": "02",
    "02002": "02",
    "02003": "02",
    "02004": "02",
    "02005": "02",
    "02006": "02",
    "02007": "02",
    "02100": "02",
    "02101": "02",
    "02102": "02",
    "03000": "03",
    "03001": "03",
    "03002": "03",
    "03003": "03",
    "03004": "03",
    "03005": "03",
    "03006": "03",
    "03007": "03",
    "03008": "03",
    "03009": "03",
    "03010": "03",
    "03011": "03",
    "03012": "03",
    "03013": "03",
    "03014": "03",
    "03015": "03",
    "03016": "03",
    "03017": "03",
    "03018": "03",
    "03019": "03",
    "03020": "03",
    "03021": "03",
    "03022": "03",
    "03023": "03",
    "03024": "03",
    "03025": "03",
    "03026": "03",
    "03027": "03",
    "03028": "03",
    "03029": "03",
    "03030": "03",
    "03031": "03",
    "03032": "03",
    "03033": "03",
    "03034": "03",
    "03035": "03",
    "03036": "03",
    "03037": "03",
    "03038": "03",
    "03100": "03",
    "03101": "03",
    "03102": "03",
    "03103": "03",
    "03104": "03",
    "03105": "03",
    "03106": "03",
    "03107": "03",
    "03108": "03",
    "03109": "03",
    "03110": "03",
    "03111": "03",
    "03112": "03",
    "03113": "03",
    "03114": "03",
    "03115": "03",
    "03199": "03",
    "03200": "03",
    "03201": "03",
    "03202": "03",
    "03203": "03",
    "03204": "03",
    "03205": "03",
    "03299": "03",
    "03300": "03",
    "03999": "03",
    "04000": "04",
    "04001": "04",
    "04002": "04",
    "04003": "04",
    "04004": "04",
    "04005": "04",
    "04006": "04",
    "04007": "04",
    "04008": "04",
    "04009": "04",
    "04010": "04",
    "04011": "04",
    "04012": "04",
    "04013": "04",
    "04014": "04",
    "04015": "04",
    "04016": "04",
    "04017": "04",
    "04018": "04",
    "04019": "04",
    "04020": "04",
    "04021": "04",
    "04022": "04",
    "04023": "04",
    "04024": "04",
    "04025": "04",
    "04026": "04",
    "04027": "04",
    "04028": "04",
    "04029": "04",
    "04030": "04",
    "04031": "04",
    "04032": "04",
    "04033": "04",
    "04034": "04",
    "04035": "04",
    "04036": "04",
    "04037": "04",
    "04038": "04",
    "04039": "04",
    "04040": "04",
    "04041": "04",
    "04042": "04",
    "04043": "04",
    "04044": "04",
    "04045": "04",
    "04046": "04",
    "04047": "04",
    "04048": "04",
    "04049": "04",
    "04050": "04",
    "04100": "04",
    "04101": "04",
    "04102": "04",
    "04103": "04",
    "04104": "04",
    "04105": "04",
    "04106": "04",
    "04107": "04",
    "04108": "04",
    "04109": "04",
    "04199": "04",
    "04200": "04",
    "04201": "04",
    "04202": "04",
    "04203": "04",
    "04204": "04",
    "04205": "04",
    "04206": "04",
    "04207": "04",
    "04208": "04",
    "04299": "04",
    "04300": "04",
    "04301": "04",
    "04302": "04",
    "04303": "04",
    "04304": "04",
    "04305": "04",
    "04306": "04",
    "04307": "04",
    "04308": "04",
    "04399": "04",
    "04400": "04",
    "04401": "04",
    "04402": "04",
    "04403": "04",
    "04404": "04",
    "04405": "04",
    "04406": "04",
    "04499": "04",
    "04999": "04",
    "05000": "05",
    "05100": "05",
    "05101": "05",
    "05102": "05",
    "05103": "05",
    "05104": "05",
    "05105": "05",
    "05106": "05",
    "05107": "05",
    "05108": "05",
    "05199": "05",
    "05200": "05",
    "05201": "05",
    "05202": "05",
    "05203": "05",
    "05204": "05",
    "05205": "05",
    "05299": "05",
    "05300": "05",
    "05301": "05",
    "05302": "05",
    "05303": "05",
    "05304": "05",
    "05305": "05",
    "05306": "05",
    "05399": "05",
    "05400": "05",
    "05401": "05",
    "05402": "05",
    "05403": "05",
    "05404": "05",
    "05405": "05",
    "05406": "05",
    "05407": "05",
    "05408": "05",
    "05499": "05",
    "05500": "05",
    "05501": "05",
    "05502": "05",
    "05503": "05",
    "05599": "05",
    "05999": "05",
    "06000": "06",
    "06100": "06",
    "06101": "06",
    "06102": "06",
    "06103": "06",
    "06104": "06",
    "06105": "06",
    "06106": "06",
    "06107": "06",
    "06108": "06",
    "06109": "06",
    "06110": "06",
    "06199": "06",
    "06999": "06",
    "07000": "07",
    "07001": "07",
    "07100": "07",
    "07101": "07",
    "07102": "07",
    "07103": "07",
    "07104": "07",
    "07105": "07",
    "07199": "07",
    "07200": "07",
    "07201": "07",
    "07202": "07",
    "07203": "07",
    "07204": "07",
    "07205": "07",
    "07206": "07",
    "07207": "07",
    "07208": "07",
    "07209": "07",
    "07210": "07",
    "07211": "07",
    "07212": "07",
    "07300": "07",
    "07301": "07",
    "07302": "07",
    "07399": "07",
    "07999": "07",
    "08000": "08",
    "08001": "08",
    "08002": "08",
    "08003": "08",
    "08004": "08",
    "08005": "08",
    "08006": "08",
    "08007": "08",
    "08008": "08",
    "08999": "08",
    "09000": "09",
    "09100": "09",
    "09101": "09",
    "09102": "09",
    "09103": "09",
    "09104": "09",
    "09105": "09",
    "09106": "09",
    "09107": "09",
    "09108": "09",
    "09109": "09",
    "09199": "09",
    "09200": "09",
    "09201": "09",
    "09202": "09",
    "09299": "09",
    "09301": "09",
    "09999": "09",
    "10000": "10",
    "10001": "10",
    "10002": "10",
    "10003": "10",
    "10004": "10",
    "10005": "10",
    "10100": "10",
    "10101": "10",
    "10102": "10",
    "10103": "10",
    "10104": "10",
    "10105": "10",
    "10106": "10",
    "10107": "10",
    "10199": "10",
    "10200": "10",
    "10201": "10",
    "10202": "10",
    "10299": "10",
    "10300": "10",
    "10301": "10",
    "10302": "10",
    "10303": "10",
    "10399": "10",
    "10400": "10",
    "10401": "10",
    "10402": "10",
    "10999": "10",
    "11000": "11",
    "11100": "11",
    "11101": "11",
    "11102": "11",
    "11103": "11",
    "11104": "11",
    "11199": "11",
    "11201": "11",
    "11202": "11",
    "11203": "11",
    "11999": "11",
    "12000": "12",
    "12001": "12",
    "12100": "12",
    "12101": "12",
    "12102": "12",
    "12103": "12",
    "12104": "12",
    "12105": "12",
    "12199": "12",
    "12200": "12",
    "12201": "12",
    "12202": "12",
    "12203": "12",
    "12299": "12",
    "12300": "12",
    "12301": "12",
    "12302": "12",
    "12303": "12",
    "12304": "12",
    "12305": "12",
    "12399": "12",
    "12999": "12",
    "13000": "13",
    "13001": "13",
    "13002": "13",
    "13003": "13",
    "13004": "13",
    "13005": "13",
    "13006": "13",
    "13007": "13",
    "13008": "13",
    "13009": "13",
    "13010": "13",
    "13011": "13",
    "13012": "13",
    "13014": "13",
    "13100": "13",
    "13101": "13",
    "13102": "13",
    "13103": "13",
    "13104": "13",
    "13105": "13",
    "13106": "13",
    "13107": "13",
    "13108": "13",
    "13109": "13",
    "13110": "13",
    "13999": "13",
    "14000": "14",
    "14100": "14",
    "14101": "14",
    "14102": "14",
    "14103": "14",
    "14199": "14",
    "14200": "14",
    "14201": "14",
    "14202": "14",
    "14203": "14",
    "14204": "14",
    "14205": "14",
    "14206": "14",
    "14299": "14",
    "15000": "15",
    "15001": "15",
    "15002": "15",
    "15003": "15",
    "15100": "15",
    "15101": "15",
    "15102": "15",
    "15103": "15",
    "15104": "15",
    "15199": "15",
    "15200": "15",
    "15201": "15",
    "15202": "15",
    "15203": "15",
    "15204": "15",
    "15205": "15",
    "15206": "15",
    "15207": "15",
    "15208": "15",
    "15299": "15",
    "15300": "15",
    "15999": "15",
    "16000": "16",
    "16001": "16",
    "16002": "16",
    "16003": "16",
    "16004": "16",
    "16005": "16",
    "16006": "16",
    "16007": "16",
    "16008": "16",
    "16009": "16",
    "16010": "16",
    "16011": "16",
    "16012": "16",
    "16013": "16",
    "16999": "16",
    "17000": "17",
    "17100": "17",
    "17200": "17",
    "17300": "17",
    "17301": "17",
    "17302": "17",
    "17303": "17",
    "17304": "17",
    "17305": "17",
    "17306": "17",
    "17307": "17",
    "17308": "17",
    "17309": "17",
    "17999": "17",
    "18000": "18",
    "18001": "18",
    "18002": "18",
    "18003": "18",
    "18004": "18",
    "18100": "18",
    "18101": "18",
    "18102": "18",
    "18103": "18",
    "18999": "18",
    "19000": "19",
    "19001": "19",
    "19002": "19",
    "19003": "19",
    "19004": "19",
    "19005": "19",
    "19006": "19",
    "19007": "19",
    "19008": "19",
    "19009": "19",
    "19010": "19",
    "19011": "19",
    "19012": "19",
    "19013": "19",
    "19014": "19",
    "19015": "19",
    "19016": "19",
    "19017": "19",
    "19018": "19",
    "19019": "19",
    "19020": "19",
    "19021": "19",
    "19022": "19",
    "19999": "19",
    "20000": "20",
    "20001": "20",
    "20002": "20",
    "20003": "20",
    "20004": "20",
    "20005": "20",
    "20006": "20",
    "20007": "20",
    "20008": "20",
    "20009": "20",
    "20010": "20",
    "20011": "20",
    "20999": "20",
    "21000": "21",
    "21001": "21",
    "21002": "21",
    "21003": "21",
    "21004": "21",
    "21005": "21",
    "21006": "21",
    "21100": "21",
    "21101": "21",
    "21102": "21",
    "21103": "21",
    "21104": "21",
    "21105": "21",
    "21106": "21",
    "21107": "21",
    "21108": "21",
    "21200": "21",
    "21201": "21",
    "21202": "21",
    "21203": "21",
    "21204": "21",
    "21205": "21",
    "21206": "21",
    "21207": "21",
    "21299": "21",
    "21999": "21",
    "22000": "22",
    "22001": "22",
    "22002": "22",
    "22003": "22",
    "22004": "22",
    "22005": "22",
    "22999": "22",
    "23000": "23",
    "23001": "23",
    "23002": "23",
    "23003": "23",
    "23004": "23",
    "23005": "23",
    "23006": "23",
    "23007": "23",
    "23100": "23",
    "23101": "23",
    "23102": "23",
    "23103": "23",
    "23200": "23",
    "23201": "23",
    "23202": "23",
    "23203": "23",
    "23999": "23",
    "24000": "24",
    "24001": "24",
    "24002": "24",
    "24003": "24",
    "24004": "24",
    "24005": "24",
    "24100": "24",
    "24101": "24",
    "24102": "24",
    "24103": "24",
    "24104": "24",
    "24105": "24",
    "24999": "24",
    "25000": "25",
    "25001": "25",
    "25999": "25",
    "26000": "26",
    "26001": "26",
    "26002": "26",
    "26003": "26",
    "26004": "26",
    "26005": "26",
    "26006": "26",
    "26100": "26",
    "26101": "26",
    "26102": "26",
    "26103": "26",
    "26999": "26",
    "27000": "27",
    "27001": "27",
    "27002": "27",
    "27003": "27",
    "27004": "27",
    "27005": "27",
    "27006": "27",
    "27007": "27",
    "27008": "27",
    "27009": "27",
    "27999": "27",
    "28000": "28",
    "28001": "28",
    "28002": "28",
    "28003": "28",
    "28004": "28",
    "28005": "28",
    "28006": "28",
    "28007": "28",
    "28008": "28",
    "28009": "28",
    "28010": "28",
    "28011": "28",
    "28012": "28",
    "28013": "28",
    "28014": "28",
    "28015": "28",
    "28016": "28",
    "28017": "28",
    "28018": "28",
    "28019": "28",
    "28020": "28",
    "28021": "28",
    "28022": "28",
    "28023": "28",
    "28999": "28",
    "29000": "29",
    "29001": "29",
    "29002": "29",
    "30000": "30",
    "30001": "30",
    "30002": "30",
    "30003": "30",
    "30004": "30",
    "30005": "30",
    "30999": "30",
    "31000": "31",
    "31001": "31",
    "31002": "31",
    "31003": "31",
    "31004": "31",
    "31005": "31",
    "31999": "31",
    "32000": "32",
    "32001": "32",
    "32002": "32",
    "32003": "32",
    "32004": "32",
    "32005": "32",
    "32100": "32",
    "32101": "32",
    "32102": "32",
    "32103": "32",
    "32999": "32",
    "33000": "33",
    "33001": "33",
    "33999": "33",
    "Z0000": "Z0",
    "Z3000": "Z3",
    "Z3001": "Z3",
    "Z3002": "Z3",
    "Z3003": "Z3",
    "Z3004": "Z3",
    "Z3005": "Z3",
    "Z3006": "Z3",
    "Z3007": "Z3",
    "Z3008": "Z3",
    "Z3009": "Z3",
    "Z3010": "Z3",
    "Z3011": "Z3",
    "Z3012": "Z3",
    "Z3014": "Z3",
    "Z3015": "Z3",
    "Z3016": "Z3",
    "Z3017": "Z3",
    "Z3018": "Z3",
    "Z3019": "Z3",
    "Z3020": "Z3",
    "Z3021": "Z3",
    "Z3022": "Z3",
    "Z3023": "Z3",
    "Z3100": "Z3",
    "Z3101": "Z3",
    "Z3102": "Z3",
    "Z3103": "Z3",
    "Z3104": "Z3",
    "Z3105": "Z3",
    "Z3106": "Z3",
    "Z3107": "Z3",
    "Z3108": "Z3",
    "Z3109": "Z3",
    "Z3199": "Z3",
    "Z3201": "Z3",
    "Z3202": "Z3",
    "Z3203": "Z3",
    "Z3204": "Z3",
    "Z3205": "Z3",
    "Z3206": "Z3",
    "Z3207": "Z3",
    "Z3208": "Z3",
    "Z3209": "Z3",
    "Z3210": "Z3",
    "Z3211": "Z3",
    "Z3212": "Z3",
    "Z3214": "Z3",
    "Z3215": "Z3",
    "Z3216": "Z3",
    "Z3217": "Z3",
    "Z3218": "Z3",
    "Z3219": "Z3",
    "Z3220": "Z3",
    "Z3221": "Z3",
    "Z3300": "Z3",
    "Z3301": "Z3",
    "Z3302": "Z3",
    "Z3303": "Z3",
    "Z3304": "Z3",
    "Z3305": "Z3",
    "Z3306": "Z3",
    "Z3307": "Z3",
    "Z3399": "Z3",
    "Z3400": "Z3",
    "Z3401": "Z3",
    "Z3402": "Z3",
    "Z3403": "Z3",
    "Z3499": "Z3",
    "Z3500": "Z3",
    "Z3501": "Z3",
    "Z3502": "Z3",
    "Z3503": "Z3",
    "Z3599": "Z3",
    "Z3600": "Z3",
    "Z3601": "Z3",
    "Z3602": "Z3",
    "Z3603": "Z3",
    "Z3604": "Z3",
    "Z3699": "Z3",
    "Z3999": "Z3",
    "Z5000": "Z5",
    "Z6000": "Z6",
    "Z6205": "Z6",
    "Z6999": "Z6",
    "Z7001": "Z7",
    "Z7002": "Z7",
    "Z8000": "Z8",
    "Z8999": "Z8",
    "Z9000": "Z9",
    "Z9001": "Z9",
    "Z9999": "Z9",
}

건축법 제2조 제1항 제11호 나목

시장ㆍ군수ㆍ구청장(자치구의 구청장을 말한다. 이하 같다)

제11조(건축허가) ① 건축물을 건축하거나 대수선하려는 자는 특별자치시장ㆍ특별자치도지사 또는 시장ㆍ군수ㆍ구청장의 허가를 받아야 한다.


자치구가 아닌 구를 제외한 집계단위는 나중에 따로 사용


In [42]:
sigungu_autonomous_agg = {
    "11110": "11110",
    "11140": "11140",
    "11170": "11170",
    "11200": "11200",
    "11215": "11215",
    "11230": "11230",
    "11260": "11260",
    "11290": "11290",
    "11305": "11305",
    "11320": "11320",
    "11350": "11350",
    "11380": "11380",
    "11410": "11410",
    "11440": "11440",
    "11470": "11470",
    "11500": "11500",
    "11530": "11530",
    "11545": "11545",
    "11560": "11560",
    "11590": "11590",
    "11620": "11620",
    "11650": "11650",
    "11680": "11680",
    "11710": "11710",
    "11740": "11740",
    "26110": "26110",
    "26140": "26140",
    "26170": "26170",
    "26200": "26200",
    "26230": "26230",
    "26260": "26260",
    "26290": "26290",
    "26320": "26320",
    "26350": "26350",
    "26380": "26380",
    "26410": "26410",
    "26440": "26440",
    "26470": "26470",
    "26500": "26500",
    "26530": "26530",
    "26710": "26710",
    "27110": "27110",
    "27140": "27140",
    "27170": "27170",
    "27200": "27200",
    "27230": "27230",
    "27260": "27260",
    "27290": "27290",
    "27710": "27710",
    "28110": "28110",
    "28140": "28140",
    "28170": "28177",
    "28177": "28177",
    "28185": "28185",
    "28200": "28200",
    "28237": "28237",
    "28245": "28245",
    "28260": "28260",
    "28710": "28710",
    "28720": "28720",
    "29110": "29110",
    "29140": "29140",
    "29155": "29155",
    "29170": "29170",
    "29200": "29200",
    "30110": "30110",
    "30140": "30140",
    "30170": "30170",
    "30200": "30200",
    "30230": "30230",
    "31110": "31110",
    "31140": "31140",
    "31170": "31170",
    "31200": "31200",
    "31710": "31710",
    "36110": "36110",
    "41111": "41110",
    "41113": "41110",
    "41115": "41110",
    "41117": "41110",
    "41131": "41130",
    "41133": "41130",
    "41135": "41130",
    "41150": "41150",
    "41171": "41170",
    "41173": "41170",
    "41190": "41190",
    "41210": "41210",
    "41220": "41220",
    "41250": "41250",
    "41271": "41270",
    "41273": "41270",
    "41281": "41280",
    "41283": "41280",
    "41285": "41280",
    "41287": "41280",
    "41290": "41290",
    "41310": "41310",
    "41360": "41360",
    "41370": "41370",
    "41390": "41390",
    "41410": "41410",
    "41430": "41430",
    "41450": "41450",
    "41461": "41460",
    "41463": "41460",
    "41465": "41460",
    "41480": "41480",
    "41500": "41500",
    "41550": "41550",
    "41570": "41570",
    "41590": "41590",
    "41610": "41610",
    "41630": "41630",
    "41650": "41650",
    "41670": "41670",
    "41800": "41800",
    "41820": "41820",
    "41830": "41830",
    "42110": "42110",
    "42130": "42130",
    "42150": "42150",
    "42170": "42170",
    "42190": "42190",
    "42210": "42210",
    "42230": "42230",
    "42720": "42720",
    "42730": "42730",
    "42750": "42750",
    "42760": "42760",
    "42770": "42770",
    "42780": "42780",
    "42790": "42790",
    "42800": "42800",
    "42810": "42810",
    "42820": "42820",
    "42830": "42830",
    "43111": "43110",
    "43112": "43110",
    "43113": "43110",
    "43114": "43110",
    "43130": "43130",
    "43150": "43150",
    "43720": "43720",
    "43730": "43730",
    "43740": "43740",
    "43745": "43745",
    "43750": "43750",
    "43760": "43760",
    "43770": "43770",
    "43800": "43800",
    "44130": "44130",
    "44131": "44130",
    "44133": "44130",
    "44150": "44150",
    "44180": "44180",
    "44200": "44200",
    "44210": "44210",
    "44230": "44230",
    "44250": "44250",
    "44270": "44270",
    "44710": "44710",
    "44760": "44760",
    "44770": "44770",
    "44790": "44790",
    "44800": "44800",
    "44810": "44810",
    "44825": "44825",
    "45111": "45110",
    "45113": "45110",
    "45130": "45130",
    "45140": "45140",
    "45180": "45180",
    "45190": "45190",
    "45210": "45210",
    "45710": "45710",
    "45720": "45720",
    "45730": "45730",
    "45740": "45740",
    "45750": "45750",
    "45770": "45770",
    "45790": "45790",
    "45800": "45800",
    "46110": "46110",
    "46130": "46130",
    "46150": "46150",
    "46170": "46170",
    "46230": "46230",
    "46710": "46710",
    "46720": "46720",
    "46730": "46730",
    "46770": "46770",
    "46780": "46780",
    "46790": "46790",
    "46800": "46800",
    "46810": "46810",
    "46820": "46820",
    "46830": "46830",
    "46840": "46840",
    "46860": "46860",
    "46870": "46870",
    "46880": "46880",
    "46890": "46890",
    "46900": "46900",
    "46910": "46910",
    "47111": "47110",
    "47113": "47110",
    "47130": "47130",
    "47150": "47150",
    "47170": "47170",
    "47190": "47190",
    "47210": "47210",
    "47230": "47230",
    "47250": "47250",
    "47280": "47280",
    "47290": "47290",
    "47720": "47720",
    "47730": "47730",
    "47750": "47750",
    "47760": "47760",
    "47770": "47770",
    "47820": "47820",
    "47830": "47830",
    "47840": "47840",
    "47850": "47850",
    "47900": "47900",
    "47920": "47920",
    "47930": "47930",
    "47940": "47940",
    "48121": "48120",
    "48123": "48120",
    "48125": "48120",
    "48127": "48120",
    "48129": "48120",
    "48170": "48170",
    "48220": "48220",
    "48240": "48240",
    "48250": "48250",
    "48270": "48270",
    "48310": "48310",
    "48330": "48330",
    "48720": "48720",
    "48730": "48730",
    "48740": "48740",
    "48820": "48820",
    "48840": "48840",
    "48850": "48850",
    "48860": "48860",
    "48870": "48870",
    "48880": "48880",
    "48890": "48890",
    "50110": "50110",
    "50130": "50130",
}

하지만 주민등록인구의 시군구 인구를 확인하고 단위를 맞출 필요는 있음. 교정이 필요한 경우에만 시군구 코드 변경(일부는 err로 집계)


In [43]:
class smart_dict(dict):
    def __missing__(self, key):
        return key


sigungu_fix = smart_dict(
    {
        "28170": "28177",  # 인천시 남구 미추홀구로 바뀌었는데 일부 남음
        "41283": "err",  # 경기도 고양시 일산구 분구되었는데 일부 남음
        "44130": "err",  # 천안시 일부 남음
    }
)

sigungu_agg = {
    key: sigungu_fix[key]
    for key in df_floor.select("시군구_코드")
    .unique()
    .sort("시군구_코드")
    .collect()["시군구_코드"]
    .to_list()
}

print(sigungu_agg["11110"])
print(sigungu_agg["28170"])
print(sigungu_agg["44130"])

11110
28177
err


## 층별개요 데이터 정제


In [44]:
df_floor.columns

['대지_위치',
 '도로명_대지_위치',
 '건물_명',
 '시군구_코드',
 '법정동_코드',
 '대지_구분_코드',
 '번',
 '지',
 '특수지_명',
 '블록',
 '로트',
 '새주소_도로_코드',
 '새주소_법정동_코드',
 '새주소_지상지하_코드',
 '새주소_본_번',
 '새주소_부_번',
 '동_명',
 '층_구분_코드',
 '층_구분_코드_명',
 '층_번호',
 '층_번호_명',
 '구조_코드',
 '구조_코드_명',
 '기타_구조',
 '주_용도_코드',
 '주_용도_코드_명',
 '기타_용도',
 '면적(㎡)',
 '주_부속_구분_코드',
 '주_부속_구분_코드_명',
 '면적_제외_여부',
 '생성_일자',
 '관리_건축물대장_PK']

In [45]:
df_floor_selected = df_floor.select(
    [
        "시군구_코드",
        "층_구분_코드",
        "층_구분_코드_명",
        "층_번호",
        "층_번호_명",
        "주_용도_코드",
        "주_용도_코드_명",
        "면적(㎡)",
        "주_부속_구분_코드",
        "주_부속_구분_코드_명",
        "면적_제외_여부",
        "생성_일자",
        "관리_건축물대장_PK",
    ]
)

시군구 코드, 층 구분 코드, 주 용도 코드, 면적 (0 초과)은 있어야 하는 것인데, 없는 경우는 제외하였다.


In [46]:
df_floor_selected.count().collect()

시군구_코드,층_구분_코드,층_구분_코드_명,층_번호,층_번호_명,주_용도_코드,주_용도_코드_명,면적(㎡),주_부속_구분_코드,주_부속_구분_코드_명,면적_제외_여부,생성_일자,관리_건축물대장_PK
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
20670074,20669516,20669516,20670074,20666793,20644000,20641697,20670074,20660591,20660591,7716626,20670074,20670074


설명: 층 구분 코드가 없으면 주 용도 코드도 없고 면적도 0이고, 하나만 하지 않는다...


In [47]:
df_floor_selected.filter(pl.col("층_구분_코드").is_null()).head().collect()

시군구_코드,층_구분_코드,층_구분_코드_명,층_번호,층_번호_명,주_용도_코드,주_용도_코드_명,면적(㎡),주_부속_구분_코드,주_부속_구분_코드_명,면적_제외_여부,생성_일자,관리_건축물대장_PK
str,str,str,i64,str,str,str,f64,str,str,str,str,str
"""11170""",null,null,0,"""(내)""",null,null,0.0,"""0""","""주건축물""","""1""","""20200427""","""11170-100188912"""
"""11170""",null,null,0,"""(내)""",null,null,0.0,"""0""","""주건축물""","""1""","""20200427""","""11170-100188913"""
"""11170""",null,null,0,"""(내)""",null,null,0.0,"""0""","""주건축물""","""1""","""20200427""","""11170-100188914"""
"""11170""",null,null,0,"""(내)""",null,null,0.0,"""0""","""주건축물""","""1""","""20200427""","""11170-100188915"""
"""11170""",null,null,0,"""(내)""",null,null,0.0,"""0""","""주건축물""","""1""","""20200427""","""11170-100188916"""


층별개요 데이터에서 시군구*코드, 층*구분*코드, 주*용도\_코드 컬럼의 값이 존재하고(null이 아니고), 면적(㎡) 컬럼의 값이 0보다 큰 유효한 경우만 추출하였다.


In [48]:
df_filtered = df_floor_selected.filter(
    pl.col("시군구_코드").is_not_null()
    & pl.col("층_구분_코드").is_not_null()
    & pl.col("주_용도_코드").is_not_null()
    & (pl.col("면적(㎡)") > 0)
)

In [49]:
# df_filtered.count().collect()

## 표제부 데이터 연계


In [50]:
df_dong.columns

['대장_구분_코드',
 '대장_구분_코드_명',
 '대장_종류_코드',
 '대장_종류_코드_명',
 '대지_위치',
 '도로명_대지_위치',
 '건물_명',
 '시군구_코드',
 '법정동_코드',
 '대지_구분_코드',
 '번',
 '지',
 '특수지_명',
 '블록',
 '로트',
 '외필지_수',
 '새주소_도로_코드',
 '새주소_법정동_코드',
 '새주소_지상지하_코드',
 '새주소_본_번',
 '새주소_부_번',
 '동_명',
 '주_부속_구분_코드',
 '주_부속_구분_코드_명',
 '대지_면적(㎡)',
 '건축_면적(㎡)',
 '건폐_율(%)',
 '연면적(㎡)',
 '용적_률_산정_연면적(㎡)',
 '용적_률(%)',
 '구조_코드',
 '구조_코드_명',
 '기타_구조',
 '주_용도_코드',
 '주_용도_코드_명',
 '기타_용도',
 '지붕_코드',
 '지붕_코드_명',
 '기타_지붕',
 '세대_수(세대)',
 '가구_수(가구)',
 '높이(m)',
 '지상_층_수',
 '지하_층_수',
 '승용_승강기_수',
 '비상용_승강기_수',
 '부속_건축물_수',
 '부속_건축물_면적(㎡)',
 '총_동_연면적(㎡)',
 '옥내_기계식_대수(대)',
 '옥내_기계식_면적(㎡)',
 '옥외_기계식_대수(대)',
 '옥외_기계식_면적(㎡)',
 '옥내_자주식_대수(대)',
 '옥내_자주식_면적(㎡)',
 '옥외_자주식_대수(대)',
 '옥외_자주식_면적(㎡)',
 '허가_일',
 '착공_일',
 '사용승인_일',
 '허가번호_년',
 '허가번호_기관_코드',
 '허가번호_기관_코드_명',
 '허가번호_구분_코드',
 '허가번호_구분_코드_명',
 '호_수(호)',
 '에너지효율_등급',
 '에너지절감_율',
 '에너지_EPI점수',
 '친환경_건축물_등급',
 '친환경_건축물_인증점수',
 '지능형_건축물_등급',
 '지능형_건축물_인증점수',
 '생성_일자',
 '내진_설계_적용_여부',
 '내진_능력',
 '관리_건축물대장_PK']

In [51]:
df_dong_selected = df_dong.select(
    [
        "시군구_코드",
        "주_부속_구분_코드",
        "주_부속_구분_코드_명",
        "대지_면적(㎡)",
        "건축_면적(㎡)",
        "연면적(㎡)",
        "용적_률_산정_연면적(㎡)",
        "주_용도_코드",
        "주_용도_코드_명",
        "사용승인_일",
        "관리_건축물대장_PK",
    ]
)

1. 전체 일반건축물/집합건축물의 표제부 동 수: 7,943,168 동
2. 주건축물 7,308,393 동, 부속건축물 634,775 동
3. 연면적 기재 7,879,427 동


In [52]:
df_dong_selected.count().collect()

시군구_코드,주_부속_구분_코드,주_부속_구분_코드_명,대지_면적(㎡),건축_면적(㎡),연면적(㎡),용적_률_산정_연면적(㎡),주_용도_코드,주_용도_코드_명,사용승인_일,관리_건축물대장_PK
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
7943168,7942481,7942481,7943168,7943168,7943168,7943168,7912311,7912173,7270272,7943168


In [53]:
df_dong_selected.select("주_부속_구분_코드").group_by(
    "주_부속_구분_코드"
).len().collect()

주_부속_구분_코드,len
str,u32
null,687
"""1""",634775
"""0""",7307706


연면적에 (0이 아닌) 값이 가장 많기 때문에 연면적을 기준으로 정제 (계획)


In [54]:
df_dong_selected.select(
    [
        (pl.col("대지_면적(㎡)") > 0).sum().alias("대지_면적(㎡) > 0"),
        (pl.col("건축_면적(㎡)") > 0).sum().alias("건축_면적(㎡) > 0"),
        (pl.col("연면적(㎡)") > 0).sum().alias("연면적(㎡) > 0"),
        (pl.col("용적_률_산정_연면적(㎡)") > 0)
        .sum()
        .alias("용적_률_산정_연면적(㎡) > 0"),
    ]
).collect()

대지_면적(㎡) > 0,건축_면적(㎡) > 0,연면적(㎡) > 0,용적_률_산정_연면적(㎡) > 0
u32,u32,u32,u32
4160181,7452568,7879427,7462902


표제부 기준, 2022년 말 이전 사용승인된 주건축물만 추출: 7,307,806 동


In [55]:
df_dong_filtered = (
    df_dong_selected.filter(
        (pl.col("주_부속_구분_코드") == "0") | (pl.col("주_부속_구분_코드").is_null())
    )
    .filter(
        (pl.col("사용승인_일").str.len_chars() != 8)
        | (pl.col("사용승인_일").fill_null("19000101") <= "20221231")
    )
    .select(["연면적(㎡)", "관리_건축물대장_PK"])
)

In [56]:
df_dong_filtered.count().collect()

연면적(㎡),관리_건축물대장_PK
u32,u32
7307806,7307806


join


In [57]:
df_joined = df_filtered.join(df_dong_filtered, on="관리_건축물대장_PK", how="inner")

In [58]:
df_joined.select(pl.len()).collect()

len
u32
19890013


In [59]:
df_joined.filter(
    (pl.col("면적(㎡)") > pl.col("연면적(㎡)")) & (pl.col("연면적(㎡)") > 0)
).select(pl.len()).collect()

len
u32
9661


In [60]:
# df_joined.filter((pl.col("면적(㎡)") > pl.col("연면적(㎡)"))).head().collect()

연면적 기준 정제(실행): 층별면적이 연면적보다 작은 것만 남기는 것을 원칙으로 하지만, 연면적이 0 이하인 오류인 경우 제거하지 않음.


In [61]:
df_joined2 = df_joined.filter(
    (pl.col("면적(㎡)") <= pl.col("연면적(㎡)")) | (pl.col("연면적(㎡)") <= 0)
)

-   전체 층별개요 데이터: 20,613,365 층
-   2022년 말 전국 건축물 층 수: 19,890,013 층
-   층 면적이 연면적보다 큰 경우(연면적 값이 존재하는 경우에 한정): 9,661 층
-   주건축물의 유효한 층 수: 19,880,352 층


In [62]:
df_joined2.select(pl.len()).collect()

len
u32
19880352


2022년 말 기준 전국 건축물의 지상 층수의 합계는 18,056,700개 층이며, 지하 층수의 합계는 1,415,933개 층이다. 기타로 옥탑은 407,556개가 있으며, 복수층(상층)은 5개, 복수층(하층)은 6개, 각층은 152개가 있다. 이를 모두 합한 전체 층의 합계는 19,880,352개 층이다.


In [63]:
df_joined2.group_by(["층_구분_코드", "층_구분_코드_명"]).len().collect()

층_구분_코드,층_구분_코드_명,len
str,str,u32
"""20""","""지상""",18056700
"""10""","""지하""",1415933
"""22""","""복수층(상층)""",5
"""30""","""옥탑""",407556
"""21""","""복수층(하층)""",6
"""40""","""각층""",152


## 집계


In [64]:
df_sgg.head().collect()

시군구코드,시군구명,폐지여부
str,str,str
"""11000""","""서울특별시""","""존재"""
"""11110""","""서울특별시 종로구""","""존재"""
"""11140""","""서울특별시 중구""","""존재"""
"""11170""","""서울특별시 용산구""","""존재"""
"""11200""","""서울특별시 성동구""","""존재"""


In [65]:
df_joined2.columns

['시군구_코드',
 '층_구분_코드',
 '층_구분_코드_명',
 '층_번호',
 '층_번호_명',
 '주_용도_코드',
 '주_용도_코드_명',
 '면적(㎡)',
 '주_부속_구분_코드',
 '주_부속_구분_코드_명',
 '면적_제외_여부',
 '생성_일자',
 '관리_건축물대장_PK',
 '연면적(㎡)']

replace 메소드의 default 인자 사용하면 됨. 따로 만들어 쓸 필요는 없음.


In [66]:
# Function to map with a default value
# def map_with_default(value, mapping, default):
#     return mapping.get(value, default)

시군구 오류 수정, 용도 4대 용도 + 기타로 집계.


In [67]:
df_joined3 = df_joined2.select(
    ["시군구_코드", "주_용도_코드", "주_용도_코드_명", "면적(㎡)"]
).with_columns(
    pl.col("시군구_코드").replace(sigungu_agg, default="err").alias("시군구_집계단위"),
    pl.col("주_용도_코드").replace(use_agg, default="기타").alias("용도_집계단위"),
)

df_joined3.select(pl.len()).collect()

len
u32
19880352


In [68]:
df_joined3.filter(pl.col("시군구_코드") == "41111").head().collect()

시군구_코드,주_용도_코드,주_용도_코드_명,면적(㎡),시군구_집계단위,용도_집계단위
str,str,str,f64,str,str
"""41111""","""02003""","""다세대주택""",99.6,"""41111""","""02"""
"""41111""","""02003""","""다세대주택""",99.6,"""41111""","""02"""
"""41111""","""02003""","""다세대주택""",99.6,"""41111""","""02"""
"""41111""","""02003""","""다세대주택""",99.6,"""41111""","""02"""
"""41111""","""06101""","""교회""",367.29,"""41111""","""06"""


In [69]:
df_aggregated = df_joined3.group_by(["시군구_집계단위", "용도_집계단위"]).agg(
    pl.sum("면적(㎡)").cast(pl.Int64).alias("면적_합계(㎡)")
)

In [ ]:
df_pivot = (
    df_aggregated.collect()
    .pivot(values="면적_합계(㎡)", index="시군구_집계단위", columns="용도_집계단위")
    .join(
        df_sgg.select(["시군구코드", "시군구명"]).collect(),
        left_on="시군구_집계단위",
        right_on="시군구코드",
        how="left",
    )
    .sort("시군구_집계단위")
    .select(
        "시군구_집계단위",
        "시군구명",
        *list(
            df_aggregated.select(pl.col("용도_집계단위").unique().sort())
            .collect()
            .to_series()
        ),
    )
)
df_pivot.filter(pl.col("시군구_집계단위").str.starts_with("41"))

시군구_집계단위,시군구명,01,02,03,04,05,06,07,08,09,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,Z3,Z5,Z6,Z7,Z8,Z9,기타
str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""41111""","""경기도 수원시 장안구""",1938148,7388603,709481,925972,61331,158778,232272,9838,68821,1269891,193555,5507,125628,285780,65557,13915,206243,24838,12746,92727,14091,101,665,7873,null,null,676,null,null,106,null,16389,null,null,null,1553,null,null
"""41113""","""경기도 수원시 권선구""",2177441,10763922,1409812,1568516,97884,130459,574772,32828,91093,887497,160061,10457,43099,758241,115193,18817,1352945,113872,21766,1230929,66885,6346,5391,24785,4496,null,338,null,186,3732,319,1941,991,21319,null,20626,null,null
"""41115""","""경기도 수원시 팔달구""",1617451,4561137,1067093,1382557,168043,128677,312395,1110,232313,536516,100388,2361,54082,1139414,625338,66509,15679,9954,5714,356541,600,3780,null,54283,12400,525,282,null,null,null,null,233,19838,177173,null,9459,null,null
"""41117""","""경기도 수원시 영통구""",1426949,11254633,960369,1415163,94999,142426,656491,1086,230509,2745218,176900,1457,115885,1855597,71848,10500,2193054,41070,14717,467649,607,21474,null,46451,null,11814,13555,null,5754,1832,null,123183,null,3810,null,25388,15,null
"""41131""","""경기도 성남시 수정구""",2765937,4350328,989430,1351040,48985,105608,284619,6828,184087,1126016,92708,18290,66898,897226,117910,23244,268007,16471,8842,298750,12235,16642,156506,11796,null,null,6040,null,null,696,108,null,null,12726,null,61551,null,357
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""41650""","""경기도 포천시""",2301941,2660894,1326548,1645723,136055,171497,110613,336,55628,735252,175467,32703,153109,80631,305945,21927,6402920,791190,54158,107074,2332886,44197,224336,11691,54714,1765,8289,null,2414,99792,13441,35037,null,617,null,18857,null,20469
"""41670""","""경기도 여주시""",2585234,2174237,629316,919966,78599,172236,117603,10056,61978,458415,152397,35436,232381,118581,142479,9823,1473269,1875229,29233,44178,1328265,22913,38723,11431,2239,2128,16698,29,7837,17686,526,4403,null,2824,1248,38623,null,4126
"""41800""","""경기도 연천군""",918439,489344,318803,293073,38433,32016,6074,2163,20215,141316,40079,20700,29221,20234,60766,12692,368194,292249,19458,28560,1192527,34354,290820,57708,4738,1008,2351,null,null,28232,3676,3290,null,123,null,1096,null,null


주거, 상업, 공업, 교육및사회 용도는 얼추 비슷하게 나오나, 기타 용도는 크게 차이가 나기도 함.


In [71]:
df_pivot.write_csv("output/floor_area_by_sgg_orig_and_use30_agg.csv", include_bom=True)

In [ ]:
# from typing import Union


# def floor_area_by_sgg_use(df: pl.DataFrame, sigungu_agg, use_agg) -> pl.DataFrame:
#     df = df.select(
#         ["시군구_코드", "주_용도_코드", "주_용도_코드_명", "면적(㎡)"]
#     ).with_columns(
#         pl.col("시군구_코드")
#         .replace(sigungu_agg, default="err")
#         .alias("시군구_집계단위"),
#         pl.col("주_용도_코드").replace(use_agg, default="기타").alias("용도_집계단위"),
#     )
#     df_aggregated = df.group_by(["시군구_집계단위", "용도_집계단위"]).agg(
#         pl.sum("면적(㎡)").cast(pl.Int64).alias("면적_합계(㎡)")
#     )
#     df_pivot = (
#         df_aggregated.collect()
#         .pivot(values="면적_합계(㎡)", index="시군구_집계단위", columns="용도_집계단위")
#         .join(
#             df_sgg.select(["시군구코드", "시군구명"]).collect(),
#             left_on="시군구_집계단위",
#             right_on="시군구코드",
#             how="left",
#         )
#         .sort("시군구_집계단위")
#         .select(
#             "시군구_집계단위",
#             "시군구명",
#             "주거용",
#             "상업용",
#             "공업용",
#             "교육및사회용",
#             "기타",
#         )
#     )
#     return df_pivot


# floor_area_by_sgg_use(df_joined2, sigungu_autonomous_agg, use_agg).write_csv(
#     "output/floor_area_by_sgg_autonomous_and_use_agg.csv", include_bom=True
# )


